In [ ]:
from pyspark.sql import functions as F, Window

In [ ]:
import os
import glob

os.environ["HADOOP_HOME"] = "D:\\hadoop"
os.environ["PATH"] += os.pathsep + "D:\\hadoop\\bin"

In [ ]:
def latest(df, key, order_col):
    w = Window.partitionBy(key).orderBy(F.col(order_col).desc())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")

In [10]:
def clean_customers(customers):
    return (
        customers.dropDuplicates(["customer_id"])
        .withColumn("city", F.coalesce(F.col("city"), F.lit("Unknown")))
    )

In [ ]:
def clean_orders(orders, products):
    o = latest(orders.dropDuplicates(), "order_id", "updated_at")
    o = o.fillna({"customer_id": 0})
    known = products.select("product_id", F.lit(True).alias("_known"))
    o = o.join(F.broadcast(known), "product_id", "left")
    reason = (F.when(F.col("quantity").isNull() | (F.col("quantity") <= 0), "bad quantity")
              .when(F.col("_known").isNull(), "unknown product"))
    o = o.withColumn("reject_reason", reason).drop("_known")
    good = o.where(F.col("reject_reason").isNull()).drop("reject_reason")
    rejected = o.where(F.col("reject_reason").isNull())
    
    return good, rejected

In [ ]:
# from spark_session import get_spark
# spark = get_spark("cleaning")
# import sources
# products = sources.products(spark)

In [ ]:
# products.show(5)

In [ ]:
# orders = sources.orders(spark)
# orders.show(5)

In [ ]:
# cleanorders_df, clean_products = clean_orders(orders, products)
# cleanorders_df.show(5)
# clean_products.show(5)
